# Day 3: 在 JupyterLab 中观察 Q 的形状变化

选择 `Python 3 (ipykernel)` 内核。先运行初始化单元格，再启用右上角的调试开关。

这份 notebook 使用 CPU 小模型和随机权重，不下载模型。先练习单元格断点，再运行仓库模型的完整验证。

In [ ]:
import os
import sys
from pathlib import Path

root = Path.cwd()
if root.name == "learning":
    root = root.parent
if not (root / "reasoning_from_scratch/qwen3.py").is_file():
    raise RuntimeError("请从仓库根目录或 learning 目录启动此 notebook。")
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import torch
from learning.day03_shapes import TINY_CONFIG, trace_model
from reasoning_from_scratch.qwen3 import Qwen3Model

torch.manual_seed(123)
model = Qwen3Model(TINY_CONFIG).eval()
input_ids = torch.arange(8).unsqueeze(0)
att = model.trf_blocks[0].att
with torch.inference_mode():
    x = model.trf_blocks[0].norm1(model.tok_emb(input_ids))
print("root:", root)
print("input:", tuple(x.shape))

## 1. 在单元格里打断点

启用右上角调试开关，在下面 `split = projected.view(...)` 那行左侧的断点栏单击，看到红点后运行函数定义单元格，再运行调用单元格。

暂停后，`split` 这一行还没有执行。查看 Variables 中的 `projected`，它应是 `[1,8,64]`。
点一次 Step Over，查看 `split`，应是 `[1,8,4,16]`；再点一次，`heads` 应是 `[1,4,8,16]`。

这里将仓库原来同一行中的 `view` / `transpose` 拆成两行，仅用于观察中间状态。后面再用真实模型的输出验证。

In [ ]:
def inspect_query(att, x):
    batch, length, _ = x.shape
    projected = att.W_query(x)
    split = projected.view(batch, length, att.num_heads, att.head_dim)
    heads = split.transpose(1, 2)
    return heads

In [ ]:
with torch.inference_mode():
    q = inspect_query(att, x)
print("Q:", tuple(q.shape))

## 2. 回到真实仓库实现验证

先点 Continue 结束上面的暂停，清除断点或关闭可视化调试，再运行以下单元格。
它直接调用仓库的 `Qwen3Model`，检查两次残差、Attention/FFN 复算和因果性，不是只验证上面的小函数。

不要在一个调试器暂停时再运行其他计算单元格；先 Continue、停止调试或重启内核。

In [ ]:
report = trace_model(model, input_ids)
for row in report["shapes"]:
    print(f'{row["name"]:38} {row["shape"]}')
print("检查数量:", len(report["checks"]))
print("结果:", set(report["checks"].values()))

## 3. 要在 qwen3.py 原始代码打断点时

在新的代码单元格运行下面这条 IPython magic（不要加 `!`）：

```python
%run -d -b reasoning_from_scratch/qwen3.py:192 learning/day03_shapes.py --config tiny --device cpu
```

出现 `ipdb>` 后输入 `c` 运行到断点，然后逐条输入：

```text
p tuple(queries.shape)
n
p tuple(queries.shape)
q
```

分别观察 `(1,8,64)` 和 `(1,4,8,16)`。这是文本调试，和上面的可视化调试二选一使用。
`q` 停止本次脚本；不会停止 Jupyter 服务。